# Decoding: From a Distribution to Text

The runnable code of [Decoding: From a Distribution to Text](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/decoding-from-logits-to-text.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 40 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB). On a GPU other than the post's Apple M4, the 16-bit rounding differences in the batching experiment will differ.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "decoding-from-logits-to-text"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/decoding-from-logits-to-text/det_check.py', 'shared/decoding-from-logits-to-text/lm.py', 'shared/decoding-from-logits-to-text/sampling.py', 'data/prompts.json']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The policies, as code

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/#the-policies-as-code)

In [ ]:
import torch

# Decoding policies, each a function from the model's logits for the next
# token to a choice. They're written as edits to the logits (masking a token
# means setting its logit to -inf), applied in a fixed order, then one draw.

def repetition_penalty(logits, history, theta):
    """Repetition penalty (Keskar et al., 2019): every token already generated
    becomes less likely. CTRL divides its logit by theta; dividing a negative
    logit would raise it, so (as in Hugging Face) negative ones are multiplied."""
    if theta == 1.0 or not history:
        return logits
    seen = torch.tensor(sorted(set(history)), device=logits.device)
    l = logits[seen]
    logits[seen] = torch.where(l > 0, l / theta, l * theta)
    return logits

def frequency_presence(logits, history, frequency, presence):
    """The OpenAI-style penalties: subtract frequency * (times the token has
    appeared) and presence * (1 if it has appeared at all)."""
    if not history or (frequency == 0 and presence == 0):
        return logits
    counts = torch.bincount(torch.tensor(history, device=logits.device), minlength=logits.numel()).float()
    return logits - frequency * counts - presence * (counts > 0).float()

def top_k(logits, k):
    """Keep the k most likely tokens."""
    if k is None:
        return logits
    kth = logits.topk(k).values[-1]
    return logits.masked_fill(logits < kth, float("-inf"))

def top_p(logits, p):
    """Nucleus sampling (Holtzman et al., 2020): keep the smallest set of the
    most likely tokens whose probabilities add up to at least p."""
    if p is None:
        return logits
    sorted_logits, order = logits.sort(descending=True)
    cumulative = sorted_logits.softmax(-1).cumsum(-1)
    drop = cumulative - sorted_logits.softmax(-1) >= p           # the mass *before* this token already reached p
    return logits.masked_fill(drop.scatter(0, order, drop), float("-inf"))

def min_p(logits, p_base):
    """Min-p (Nguyen et al., 2025): keep tokens with probability at least
    p_base times the top token's probability."""
    if p_base is None:
        return logits
    probs = logits.softmax(-1)
    return logits.masked_fill(probs < p_base * probs.max(), float("-inf"))

def choose(logits, history, generator, temperature=1.0, k=None, p=None, mp=None,
           rep=1.0, freq=0.0, pres=0.0):
    """One decoding step: penalties, then temperature, then the filters, then
    a draw. Temperature 0 means greedy: always the most likely token."""
    logits = logits.float().clone()
    logits = repetition_penalty(logits, history, rep)
    logits = frequency_presence(logits, history, freq, pres)
    if temperature == 0:
        return logits.argmax().item()
    logits = logits / temperature
    logits = min_p(top_p(top_k(logits, k), p), mp)
    return torch.multinomial(logits.softmax(-1), 1, generator=generator).item()

@torch.no_grad()
def generate(model, prompt_ids, n, seed=0, stop=(), **policy):
    """Generate up to n tokens with a KV cache: the prompt once, then one
    token per step ("What Generative Means" and "The KV Cache"). Stops early at any token in `stop`."""
    g = torch.Generator(device=prompt_ids.device).manual_seed(seed)
    out = model(prompt_ids[None], use_cache=True)
    history = []
    for _ in range(n):
        token = choose(out.logits[0, -1], history, g, **policy)
        history.append(token)
        if token in stop:
            break
        out = model(torch.tensor([[token]], device=prompt_ids.device),
                    past_key_values=out.past_key_values, use_cache=True)
    return history

In [ ]:
import json
import math
import torch
from lm import model, tok, encode
from sampling import top_k, top_p, min_p

# Two real next-token distributions: one where the model is nearly sure,
# one where many continuations are reasonable. For each: the most likely
# tokens, the entropy at three temperatures, and how many tokens each
# filter keeps.
def entropy_nats(logits):
    p = logits.softmax(-1)
    return -(p * p.clamp_min(1e-30).log()).sum().item()

kept = lambda l: int((l > float("-inf")).sum())
for prompt in ("The KV cache stores the keys and", "My favorite thing about this blog is"):
    with torch.no_grad():
        logits = model(encode(prompt)[None]).logits[0, -1].float()
    p = logits.softmax(-1)
    top = p.topk(6)
    print(f"{prompt!r} ->")
    print("   " + ", ".join(f"{tok.decode(int(i))!r} {v:.3f}" for v, i in zip(top.values, top.indices)))
    print("   entropy at T = 0.5 / 1 / 1.5: " + " / ".join(f"{entropy_nats(logits / t):.2f}" for t in (0.5, 1, 1.5))
          + f" nats;  top token's probability at T = 0.1: {(logits / 0.1).softmax(-1).max():.4f}")
    print(f"   tokens kept at T = 1: top-k 40 keeps {kept(top_k(logits.clone(), 40))}, "
          f"top-p 0.9 keeps {kept(top_p(logits.clone(), 0.9))}, min-p 0.1 keeps {kept(min_p(logits.clone(), 0.1))}")
    for t in (1.5, 2.0):
        print(f"   at T = {t} (temperature applied first): top-p 0.9 keeps {kept(top_p(logits / t, 0.9))}, "
              f"min-p 0.1 keeps {kept(min_p(logits / t, 0.1))}; with the filters first, top-p's set stays "
              f"{kept(top_p(logits.clone(), 0.9))} and min-p's {kept(min_p(logits.clone(), 0.1))}")

# The open prompt's distribution, for the widget: the 200 most likely tokens
# one by one, and every other token as a histogram of logits, in bins 0.25
# wide, so the widget can still apply temperature and filters to the tail.
top = logits.topk(200)
rest = logits.sort(descending=True).values[200:]
bins = torch.floor(rest / 0.25) * 0.25 + 0.125                    # each bin's centre
centres, counts = bins.unique(return_counts=True)
print(f"the widget's 200 tokens cover {p[top.indices].sum():.1%} of the probability at T = 1; "
      f"the other {len(rest):,} tokens go in {len(centres)} bins")
json.dump({"prompt": prompt, "tokens": [tok.decode(int(i)) for i in top.indices],
           "logits": [round(v, 3) for v in top.values.tolist()],
           "tail": [[round(c, 3), int(n)] for c, n in zip(centres.tolist()[::-1], counts.tolist()[::-1])]},
          open("widget_dist.json", "w"), ensure_ascii=False)

The post printed:

```text
'The KV cache stores the keys and' ->
   ' values' 0.758, ' their' 0.084, ' corresponding' 0.043, ' the' 0.015, ' associated' 0.008, ' data' 0.008
   entropy at T = 0.5 / 1 / 1.5: 0.09 / 1.40 / 6.11 nats;  top token's probability at T = 0.1: 1.0000
   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 5, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 3655, min-p 0.1 keeps 3; with the filters first, top-p's set stays 5 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 26113, min-p 0.1 keeps 6; with the filters first, top-p's set stays 5 and min-p's 2
'My favorite thing about this blog is' ->
   ' that' 0.461, ' the' 0.211, ' how' 0.039, ' when' 0.036, ' its' 0.014, ' seeing' 0.012
   entropy at T = 0.5 / 1 / 1.5: 0.57 / 2.53 / 6.92 nats;  top token's probability at T = 0.1: 0.9996
   tokens kept at T = 1: top-k 40 keeps 40, top-p 0.9 keeps 26, min-p 0.1 keeps 2
   at T = 1.5 (temperature applied first): top-p 0.9 keeps 9542, min-p 0.1 keeps 4; with the filters first, top-p's set stays 26 and min-p's 2
   at T = 2.0 (temperature applied first): top-p 0.9 keeps 45600, min-p 0.1 keeps 20; with the filters first, top-p's set stays 26 and min-p's 2
the widget's 200 tokens cover 97.1% of the probability at T = 1; the other 151,736 tokens go in 87 bins
```

## Greedy decoding, and its loops

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/#greedy-decoding-and-its-loops)

In [ ]:
import json
import torch
from lm import model, tok, encode

# The real continuations' log-probability, prompt by prompt: how much it
# varies, and whether the paragraphs full of numbers are the hard ones.
prompts = json.load(open("prompts.json"))
rows = []
with torch.no_grad():
    for i, item in enumerate(prompts):
        prompt, ids = encode(item["prompt"]), tok(item["human"]).input_ids[:64]
        x = torch.cat([prompt, torch.tensor(ids, device=prompt.device)])
        lp = model(x[None]).logits[0, len(prompt) - 1:-1].float().log_softmax(-1)
        score = lp.gather(1, x[len(prompt):, None]).mean().item()
        numeric = sum(any(c.isdigit() for c in tok.decode([t])) for t in ids) / len(ids)
        rows.append((score, numeric, i))
rows.sort()
print("per-prompt log-prob of the real continuation: " + ", ".join(f"{s:.2f}" for s, _, _ in rows))
print(f"lowest {rows[0][0]:.2f} (prompt {rows[0][2]}, {rows[0][1]:.0%} of tokens with digits); highest {rows[-1][0]:.2f} "
      f"(prompt {rows[-1][2]}, {rows[-1][1]:.0%} with digits); mean {sum(r[0] for r in rows) / 20:.2f}")

The post printed:

```text
per-prompt log-prob of the real continuation: -4.46, -4.36, -4.24, -4.18, -4.05, -4.03, -3.91, -3.87, -3.75, -3.56, -3.43, -3.35, -3.29, -3.05, -2.99, -2.96, -2.91, -2.90, -2.74, -2.49
lowest -4.46 (prompt 10, 0% of tokens with digits); highest -2.49 (prompt 17, 39% with digits); mean -3.53
```

## Measured on twenty paragraphs

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/#measured-on-twenty-paragraphs)

In [ ]:
%%writefile step5.py
import json
import sys
import time
import torch
from lm import model, tok, encode
from sampling import generate

# Every policy on the same 20 prompts from the blog, 128 new tokens each.
# Measured: how repetitive the text is (the share of 4-token sequences that
# already occurred earlier in the same continuation), and how likely the
# model itself finds it (mean log-probability per token under its plain
# distribution, no temperature or filters). The real continuation of each
# paragraph, written by a person, is scored the same way, over its first 64
# tokens (the shortest is 67), and so are the first 64 of each policy's.
N = 128
STOP = {tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")}
prompts = json.load(open("prompts.json"))
POLICIES = {
    "greedy":                 dict(temperature=0),
    "beam search, 4 beams":   "beam",
    "greedy + repetition 1.2": dict(temperature=0, rep=1.2),
    "pure sampling, T=1":     dict(temperature=1.0),
    "T=0.7":                  dict(temperature=0.7),
    "top-k 40":               dict(k=40),
    "top-p 0.9":              dict(p=0.9),
    "min-p 0.1":              dict(mp=0.1),
    "T=1.5 alone":            dict(temperature=1.5),
    "T=1.5 + min-p 0.1":      dict(temperature=1.5, mp=0.1),
}

def repeats(ids, n=4):
    grams = [tuple(ids[i:i + n]) for i in range(len(ids) - n + 1)]
    return sum(g in set(grams[:i]) for i, g in enumerate(grams)) / max(1, len(grams))

@torch.no_grad()
def logprob(prompt, ids):
    x = torch.cat([prompt, torch.tensor(ids, device=prompt.device)])
    lp = model(x[None]).logits[0, len(prompt) - 1:-1].float().log_softmax(-1)
    return lp.gather(1, x[len(prompt):, None]).mean().item()

@torch.no_grad()
def beam(prompt):
    out = model.generate(prompt[None], max_new_tokens=N, num_beams=4, do_sample=False,
                         repetition_penalty=1.0,     # the model's generation_config.json sets 1.1
                         eos_token_id=list(STOP), pad_token_id=tok.eos_token_id)
    return out[0, len(prompt):].tolist()

if __name__ == "__main__":
    which = sys.argv[1:] or list(POLICIES)
    results = {}
    for name in which:
        policy = POLICIES[name]
        seeds = (0,) if policy == "beam" or policy.get("temperature", 1) == 0 else (0, 1)
        for seed in seeds:
            t0, texts, reps, reps64, lps = time.perf_counter(), [], [], [], []
            for item in prompts:
                prompt = encode(item["prompt"])
                ids = beam(prompt) if policy == "beam" else generate(model, prompt, N, seed=seed, stop=STOP, **policy)
                ids = [i for i in ids if i not in STOP] or [tok.eos_token_id]
                texts.append(tok.decode(ids)); reps.append(repeats(ids)); reps64.append(repeats(ids[:64]))
                lps.append(logprob(prompt, ids[:64]))
            results[f"{name}|{seed}"] = texts
            print(f"{name:24s} seed {seed}: first 64 tokens: repeated 4-grams {sum(reps64) / 20:5.1%}, "
                  f"log-prob {sum(lps) / 20:5.2f}; all 128: repeated {sum(reps) / 20:5.1%}, "
                  f"{sum(r > 0.3 for r in reps):2d}/20 over 30%  ({time.perf_counter() - t0:.0f} s)", flush=True)
    if not sys.argv[1:]:
        reps, lps = [], []
        for item in prompts:
            prompt, ids = encode(item["prompt"]), tok(item["human"]).input_ids[:64]
            reps.append(repeats(ids)); lps.append(logprob(prompt, ids))
        print(f"{'human continuation':24s}       : first 64 tokens: repeated 4-grams {sum(reps) / 20:5.1%}, "
              f"log-prob {sum(lps) / 20:5.2f}")
        json.dump(results, open("compare_texts.json", "w"), indent=1, ensure_ascii=False)

In [ ]:
%run step5.py

The post printed:

```text
greedy                   seed 0: first 64 tokens: repeated 4-grams 25.0%, log-prob -0.85; all 128: repeated 48.6%, 13/20 over 30%  (124 s)
beam search, 4 beams     seed 0: first 64 tokens: repeated 4-grams 16.8%, log-prob -0.62; all 128: repeated 41.4%, 14/20 over 30%  (131 s)
greedy + repetition 1.2  seed 0: first 64 tokens: repeated 4-grams  1.0%, log-prob -1.43; all 128: repeated  2.6%,  0/20 over 30%  (107 s)
pure sampling, T=1       seed 0: first 64 tokens: repeated 4-grams  0.5%, log-prob -3.15; all 128: repeated  1.8%,  0/20 over 30%  (105 s)
pure sampling, T=1       seed 1: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.86; all 128: repeated  3.5%,  1/20 over 30%  (101 s)
T=0.7                    seed 0: first 64 tokens: repeated 4-grams  3.4%, log-prob -1.46; all 128: repeated  5.8%,  1/20 over 30%  (109 s)
T=0.7                    seed 1: first 64 tokens: repeated 4-grams  3.8%, log-prob -1.66; all 128: repeated  8.1%,  2/20 over 30%  (112 s)
top-k 40                 seed 0: first 64 tokens: repeated 4-grams  1.2%, log-prob -2.10; all 128: repeated  2.0%,  0/20 over 30%  (113 s)
top-k 40                 seed 1: first 64 tokens: repeated 4-grams  3.4%, log-prob -1.93; all 128: repeated  5.0%,  0/20 over 30%  (107 s)
top-p 0.9                seed 0: first 64 tokens: repeated 4-grams  0.3%, log-prob -2.30; all 128: repeated  2.4%,  0/20 over 30%  (119 s)
top-p 0.9                seed 1: first 64 tokens: repeated 4-grams  3.9%, log-prob -2.11; all 128: repeated  7.0%,  1/20 over 30%  (112 s)
min-p 0.1                seed 0: first 64 tokens: repeated 4-grams  3.3%, log-prob -1.49; all 128: repeated  9.2%,  2/20 over 30%  (106 s)
min-p 0.1                seed 1: first 64 tokens: repeated 4-grams  2.9%, log-prob -1.34; all 128: repeated  8.4%,  1/20 over 30%  (113 s)
T=1.5 alone              seed 0: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.13; all 128: repeated  0.0%,  0/20 over 30%  (111 s)
T=1.5 alone              seed 1: first 64 tokens: repeated 4-grams  0.0%, log-prob -10.24; all 128: repeated  0.0%,  0/20 over 30%  (111 s)
T=1.5 + min-p 0.1        seed 0: first 64 tokens: repeated 4-grams  2.4%, log-prob -2.18; all 128: repeated  4.2%,  1/20 over 30%  (125 s)
T=1.5 + min-p 0.1        seed 1: first 64 tokens: repeated 4-grams  1.3%, log-prob -2.06; all 128: repeated  3.7%,  0/20 over 30%  (120 s)
human continuation             : first 64 tokens: repeated 4-grams  0.3%, log-prob -3.53
```

In [ ]:
import json

# One prompt's continuations from compare.py, under each policy (seed 0),
# first 260 characters, with the paragraph's real continuation last.
texts, prompts = json.load(open("compare_texts.json")), json.load(open("prompts.json"))
i = 7
print(f"PROMPT: {prompts[i]['prompt']}\n")
for name in ("greedy", "beam search, 4 beams", "greedy + repetition 1.2", "pure sampling, T=1", "top-p 0.9",
             "min-p 0.1", "T=1.5 alone", "T=1.5 + min-p 0.1"):
    print(f"[{name}]{texts[name + '|0'][i][:260]!r}\n")
print(f"[what the post says]{prompts[i]['human'][:260]!r}")

The post printed:

```text
PROMPT: All three need about the same number of updates to get close, around 50. But an update on one question costs a 24th of a full-batch update. So SGD gets there after seeing the data about twice, where

[greedy]" as the other two need to see the data about 10 times. \n\nSo, the question is, why does SGD need to see the data about 10 times, while the other two need to see the data about 50 times? \n\nI understand that SGD is a stochastic algorithm, but I don't understand w"

[beam search, 4 beams]' mini-batch gets there after seeing the data about 1.5 times. This is because mini-batch has to see the data 1.5 times to get the same number of updates as SGD, and SGD has to see the data 2 times to get the same number of updates as mini-batch. So mini-batch '

[greedy + repetition 1.2]" as the other two need to see it 3 times. I'm not sure if this is a good thing or bad, but my intuition says that SGD should be faster than both of them.\n\nI've been trying to understand why and how these things work in practice (and what they mean), so any hel"

[pure sampling, T=1]' agrad and rose have each seen almost the entire batch of updates. Worst case, I suppose. So far, by updating the golden set, I have trouble seeing a good upper bound on the number of updates it takes. Can anyone help with referencing the complexities to get t'

[top-p 0.9]' as Batch would take 15 minutes for a full-batch. \\n\\nA Batch update with a full-batch would result in a new estimate for the error after every update. If, instead, the batch update with a partial batch is used, then the error can only be updated after every $'

[min-p 0.1]' as Adam and RMSProp each take about the same amount of updates. \\n\\n'

[T=1.5 alone]' agodly rose closes in without using the heavy Smith Archive warmup quadrant.\\nTS_USD_OPEN安い decimal by:", finelist الأمم synchronization method with trouble�iers SMART công&PubOrganwoo-oriented cetверParser).\\-output[i referencing執INCLUDINGCKكف纂-language sub_'

[T=1.5 + min-p 0.1]' SGA would have seen it almost four times over.\\nI can tell you the size of my learning batches and the total number of updates needed, so you could simulate your own. Would the algorithm speed up more quickly than it would if the batch size and total updates '

[what the post says]" full-batch descent needs 52 passes. The price is noise, and it shows in the last two columns. With a constant learning rate, SGD never settles. After 5,000 updates its median loss is 0.4949 against the optimum's 0.4837, and its worst run is at 0.7688, knocked"
```

## Same prompt, same seed, different text

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/#same-prompt-same-seed-different-text)

In [ ]:
import json
import torch
from lm import model, tok, encode, DEVICE
from sampling import generate

# Is decoding reproducible? (1) The same seed twice. (2) The same prompt
# decoded greedily on its own, and inside a batch of four prompts padded to
# the same length: mathematically the same computation for this prompt.
prompts = [p["prompt"] for p in json.load(open("prompts.json"))]

same = sum(generate(model, encode(p), 64, seed=7, p=0.9) == generate(model, encode(p), 64, seed=7, p=0.9)
           for p in prompts[:5])
print(f"top-p 0.9, the same seed twice: identical text for {same} of 5 prompts")

@torch.no_grad()
def greedy(batch):
    tok.padding_side = "left"
    enc = tok(batch, return_tensors="pt", padding=True).to(DEVICE)
    out = model.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id,
                         repetition_penalty=1.0,     # the model's generation_config.json sets 1.1
                         output_scores=True, return_dict_in_generate=True)
    return out.sequences[:, enc.input_ids.shape[1]:].tolist(), out.scores[0].float()

diverged, first, gaps = 0, [], []
for i in range(0, 20, 4):
    batch = prompts[i:i + 4]
    together, scores_b = greedy(batch)
    for j, p in enumerate(batch):
        alone, scores_a = greedy([p])
        gaps.append((scores_a[0] - scores_b[j]).abs().max().item())
        a, b = alone[0], together[j]
        if a != b:
            diverged += 1
            first.append(next(k for k in range(min(len(a), len(b))) if a[k] != b[k]))
print(f"greedy, alone vs in a padded batch of 4: largest first-step logit difference "
      f"{max(gaps):.3f} (median {sorted(gaps)[10]:.3f}); the 64-token outputs differ for {diverged} of 20 "
      f"prompts, first at token {sorted(first) if first else '-'}")

The post printed:

```text
top-p 0.9, the same seed twice: identical text for 5 of 5 prompts
greedy, alone vs in a padded batch of 4: largest first-step logit difference 0.039 (median 0.027); the 64-token outputs differ for 3 of 20 prompts, first at token [22, 37, 52]
```

In [ ]:
%%writefile det_check.py
# The first step's logits for four prompts, alone and in a padded batch,
# in 16-bit and in 32-bit floats (run once with each: python det_check.py 16 / 32).
import json, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm import PATH
import torch
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"
dtype = {"16": torch.float16, "32": torch.float32}[sys.argv[1]]
tok = AutoTokenizer.from_pretrained(PATH); tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(PATH, dtype=dtype).to(GPU).eval()
prompts = [p["prompt"] for p in json.load(open("prompts.json"))][:4]
with torch.no_grad():
    enc = tok(prompts, return_tensors="pt", padding=True).to(GPU)
    pos = (enc.attention_mask.cumsum(-1) - 1).clamp(min=0)
    b = model(**enc).logits[:, -1].float()
    b2 = model(**enc, position_ids=pos).logits[:, -1].float()
    for j, p in enumerate(prompts):
        a = model(**tok([p], return_tensors="pt").to(GPU)).logits[0, -1].float()
        print(sys.argv[1], j, "pad tokens", int((enc.attention_mask[j] == 0).sum()), "diff", round((a - b[j]).abs().max().item(), 4), "with position_ids", round((a - b2[j]).abs().max().item(), 4))

In [ ]:
%run det_check.py 16
%run det_check.py 32

The post printed:

```text
16 0 pad tokens 17 diff 0.0195 with position_ids 0.0312
16 1 pad tokens 7 diff 0.0234 with position_ids 0.0234
16 2 pad tokens 0 diff 0.0273 with position_ids 0.0273
16 3 pad tokens 16 diff 0.0293 with position_ids 0.0352
32 0 pad tokens 17 diff 0.0 with position_ids 0.0
32 1 pad tokens 7 diff 0.0 with position_ids 0.0
32 2 pad tokens 0 diff 0.0 with position_ids 0.0
32 3 pad tokens 16 diff 0.0001 with position_ids 0.0001
```

## What decoding costs

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/decoding-from-logits-to-text/#what-decoding-costs)

In [ ]:
import time
import torch
from lm import model, encode
from sampling import choose
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"
gpu_sync = {"cuda": torch.cuda.synchronize, "mps": torch.mps.synchronize}.get(GPU, lambda: None)
gpu_empty_cache = {"cuda": torch.cuda.empty_cache, "mps": torch.mps.empty_cache}.get(GPU, lambda: None)

# What a decoding step costs: the model's forward pass for one new token
# (with the KV cache), against the sampling policy that picks the token.
prompt = encode("The KV cache stores the keys and values of every earlier token, so")
g = torch.Generator(device=prompt.device).manual_seed(0)
with torch.no_grad():
    out = model(prompt[None], use_cache=True)
    logits, cache = out.logits[0, -1], out.past_key_values
    def step():
        model(torch.tensor([[11]], device=prompt.device), past_key_values=cache, use_cache=True)
    def timed(fn, n=30):
        for _ in range(3):
            fn()
        gpu_sync(); t0 = time.perf_counter()
        for _ in range(n):
            fn()
        gpu_sync()
        return (time.perf_counter() - t0) / n * 1e3
    print(f"forward pass for one token: {timed(step):.1f} ms")
    for name, policy in (("greedy", dict(temperature=0)), ("top-p 0.9", dict(p=0.9)),
                         ("T 0.7 + top-k 40 + top-p 0.9 + min-p 0.05 + repetition 1.1",
                          dict(temperature=0.7, k=40, p=0.9, mp=0.05, rep=1.1))):
        print(f"choosing the token, {name}: {timed(lambda: choose(logits, list(range(50)), g, **policy)):.2f} ms")

The post printed:

```text
forward pass for one token: 41.0 ms
choosing the token, greedy: 0.24 ms
choosing the token, top-p 0.9: 0.91 ms
choosing the token, T 0.7 + top-k 40 + top-p 0.9 + min-p 0.05 + repetition 1.1: 1.61 ms
```